In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Scrie primul tău program Qiskit Serverless

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind cerințele de mai jos.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</details>
Acest exemplu demonstrează cum să folosești instrumentele `qiskit-serverless` pentru a crea un program de transpilare paralelă, și apoi să implementezi `qiskit-ibm-catalog` pentru a implementa programul tău pe IBM Quantum Platform și a-l folosi ca serviciu remote reutilizabil.
## Exemplu: transpilare la distanță cu Qiskit Serverless
Pornește cu următorul exemplu care transpilează un `circuit` față de un `backend` dat și un `optimization_level` țintă, și adaugă treptat mai multe elemente pentru a implementa volumul de lucru pe Qiskit Serverless.

Pune următoarea celulă de cod în fișierul `./source_files/transpile_remote.py`. Acesta este programul care va fi încărcat în Qiskit Serverless.

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

## Configurează-ți fișierele
Qiskit Serverless necesită organizarea fișierelor `.py` ale volumului tău de lucru într-un director dedicat. Următoarea structură este un exemplu de bună practică:

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

Serverless încarcă conținutul directorului `source_files` pentru a rula de la distanță. Odată ce acestea sunt configurate, poți ajusta `transpile_remote.py` pentru a prelua intrări și a returna ieșiri.

### Obține argumentele programului
Fișierul tău inițial `transpile_remote.py` are trei intrări: `circuits`, `backend_name` și `optimization_level`. Serverless este în prezent limitat să accepte doar intrări și ieșiri serializabile. Din acest motiv, nu poți transmite `backend` direct, deci folosește `backend_name` ca șir de caractere în schimb.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

<span id="upload-to-qiskit-serverless"></span>
### Authenticate to Qiskit Serverless

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (you can use your `QiskitRuntimeService` API key, or create a new API key on the [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com)).

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

În acest moment, poți obține Backend-ul tău cu `QiskitRuntimeService` și adăuga programul existent cu codul următor. Codul următor necesită să fi [salvat deja acreditările](/guides/cloud-setup).

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

În final, poți rula `transpile_remote()` pentru toate `circuits` transmise și returna `transpiled_circuits` ca rezultat:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

Qiskit Serverless comprimă conținutul directorului `working_dir` (în acest caz, `source_files`) într-un `tar`, care este încărcat și curățat ulterior. `entrypoint` identifică fișierul executabil principal al programului pe care Qiskit Serverless îl va rula. În plus, dacă programul tău are dependențe `pip` personalizate, le poți adăuga într-un array `dependencies`: